## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [3]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

   
Contact
4086595588 (Home)
jaydz.penuliar@gmail.com
www.linkedin.com/in/jdpenuliar
(LinkedIn)
Top Skills
PostgreSQL
Git
Docker Products
Joseph Donn Penuliar
Software Engineer at Facebook
Santa Clara, California, United States
Summary
Hey Im Joseph Donn Penuliar. People mostly call me JD. Great to
make your virtual acquaintance.
Experienced software engineer with 7 years of diverse expertise and
a goal-oriented mindset. Skilled in researching and/or developing
innovative solutions to complex problems, I have a track record of
delivering high-quality products and meeting roadmaps. Adaptable
and continuously learning, I excel at collaborating with cross-
functional teams and translating business needs into effective
development plans.
Let's get in touch:
www.jdpenuliar.com
https://linktr.ee/kassev
www.linkedin.com/in/jdpenuliar
www.github.com/jdpenuliar
jaydz.penuliar@gmail.com
Experience
Facebook
9 years 2 months
Software Engineer
August 2018 - Present (7 years 10 months)
Menlo Park, 

In [5]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()
    summary

In [6]:
summary

"My name is Ed Donner. I'm an entrepreneur, software engineer and data scientist. I'm originally from London, England, but I moved to NYC in 2000.\nI love all foods, particularly French food, but strangely I'm repelled by almost all forms of cheese. I'm not allergic, I just hate the taste! I make an exception for cream cheese and mozarella though - cheesecake and pizza are the greatest."

In [7]:
name = "JD Penuliar"

In [8]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [9]:
system_prompt

'You are acting as JD Penuliar. You are answering questions on JD Penuliar\'s website, particularly questions related to JD Penuliar\'s career, background, skills and experience. Your responsibility is to represent JD Penuliar for interactions on the website as faithfully as possible. You are given a summary of JD Penuliar\'s background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don\'t know the answer, say so.\n\n## Summary:\nMy name is Ed Donner. I\'m an entrepreneur, software engineer and data scientist. I\'m originally from London, England, but I moved to NYC in 2000.\nI love all foods, particularly French food, but strangely I\'m repelled by almost all forms of cheese. I\'m not allergic, I just hate the taste! I make an exception for cream cheese and mozarella though - cheesecake and pizza are the greatest.\n\n## LinkedIn Profile:\n\xa0 \xa0\nCon

In [ ]:
def chat(message, history):
    messages = (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": message}]
    )
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [12]:
# gr.ChatInterface(chat, type="messages").launch()

## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [31]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [40]:

reply

"As of now, I do not hold any patents. My expertise lies primarily in software engineering and data science, focusing on developing innovative solutions and delivering high-quality products. If you have any other questions or if you're interested in discussing potential collaborations, feel free to ask!"

In [32]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [33]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [62]:
import os

model_name = "claude-sonnet-4-6"
claude = OpenAI(
    api_key=os.environ.get("ANTHROPIC_API_KEY"),  # Your Claude API key
    base_url="https://api.anthropic.com/v1/",  # the Claude API endpoint
)


In [63]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [
        {"role": "system", "content": evaluator_system_prompt},
        {"role": "user", "content": evaluator_user_prompt(reply, message, history)},
    ]
    try:
        response = claude.chat.completions.parse(
            model=model_name,
            messages=messages,
            response_format=Evaluation,
            max_tokens=1000,
        )
        return response.choices[0].message.parsed
    except Exception as e:
        # Avoid crashing the chat UI when structured parsing is blocked (e.g. content filter).
        return Evaluation(
            is_acceptable=False,
            feedback=f"Evaluator unavailable ({type(e).__name__}): {e}",
        )

In [64]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=False, feedback="The agent mentioned 'data science' as part of JD's expertise, but there is no mention of data science in JD Penuliar's LinkedIn profile or summary. The agent appears to have conflated details from the summary (which describes Ed Donner, not JD Penuliar). The core answer about patents is fine, but the characterization of expertise should stick to what's in JD's actual profile (software engineering, full stack development, etc.).")

In [65]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [66]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [67]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Failed evaluation - retrying
Evaluator unavailable (ContentFilterFinishReasonError): Could not parse response content as the request was rejected by the content filter
